# End to End Deep Learning Project With Simple RNN

## Part 1: Load the IMDB dataset and prepare it

50,000 movie reviews, each labelled positive (1) or negative (0). Tested with **TensorFlow 2.21** (Keras 3).

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow version: {tf.__version__}')

TensorFlow version: 2.21.0


## 1. Load the data

`num_words` is our vocabulary size: keep only the 10,000 most frequent words.

In [2]:
max_features = 10000   # vocabulary size

# num_words: keep only the 10,000 most frequent words; rarer words are replaced by index 2 (unknown word)
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

print(f'Training data: {X_train.shape}, labels: {y_train.shape}')
print(f'Testing data : {X_test.shape}, labels: {y_test.shape}')

Training data: (25000,), labels: (25000,)
Testing data : (25000,), labels: (25000,)


In [3]:
# is the data balanced?
print(f'train  negative, positive: {np.bincount(y_train)}')
print(f'test   negative, positive: {np.bincount(y_test)}')

train  negative, positive: [12500 12500]
test   negative, positive: [12500 12500]


## 2. Look at one review

The reviews are already converted to word indexes.

In [4]:
sample_review = X_train[0]
sample_label = y_train[0]

print(f'Review as integers: {sample_review}')
print(f'Label: {sample_label}')

Review as integers: [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Label: 1


## 3. Decode it back to words (just for understanding)

IMDB keeps the first indexes for special tokens: 0 = padding, 1 = start of review, 2 = unknown word, 3 = unused.
Real words start at 4, so we subtract 3 before looking a word up.

In [5]:
word_index = imdb.get_word_index()      # word -> index
reverse_word_index = {index: word for word, index in word_index.items()}   # index -> word
# print(reverse_word_index)
# print(f'{len(word_index)} words')

print(word_index["fantastic"])
print(reverse_word_index[774])
# sample_words = {w: word_index[w] for w in ['the', 'and', 'movie', 'boring', 'brilliant']}
# print(f'{sample_words}')

774
fantastic


In [6]:
decoded_review = ' '.join(reverse_word_index.get(i - 3, '?') for i in sample_review)
decoded_review

"? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you th

The first `?` is the start token (index 1).

## 4. How long are the reviews?

In [7]:
lengths = np.array([len(r) for r in X_train])
print(f'shortest: {lengths.min()}')
print(f'average : {lengths.mean():.1f}')
print(f'median  : {np.median(lengths)}')
print(f'longest : {lengths.max()}')
print(f'longer than 500 words: {(lengths > 500).mean() * 100:.1f} %')

shortest: 11
average : 238.7
median  : 178.0
longest : 2494
longer than 500 words: 8.4 %


## 5. Pad every review to the same length

`maxlen=500`: shorter reviews get zeros at the start (`padding='pre'`, the default). Longer reviews are cut, and by default the **start** is cut (`truncating='pre'`), so the ending is kept.

In [8]:
max_len = 500

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

print(f'{X_train.shape} {X_test.shape}')

(25000, 500) (25000, 500)


In [22]:
X_train[1059]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

The data is ready: every review is now exactly 500 numbers.

## 6. Build the model

Embedding (10,000 words x 128 numbers) -> SimpleRNN (128 units) -> Dense (1 output, sigmoid).
SimpleRNN uses **tanh** by default. (For many classes: `Dense(n_classes, activation='softmax')`.)

In [9]:
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(max_features, 128),         # word id -> 128 numbers
    SimpleRNN(128),                       # tanh is the default activation
    Dense(1, activation='sigmoid'),       # probability of positive
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

## 7. Compile

`clipnorm=1.0` is gradient clipping: if the gradient gets too big, it is scaled down before the update. This protects the RNN from exploding gradients.

In [11]:
model.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss='binary_crossentropy',           # multi-class: 'sparse_categorical_crossentropy'
    metrics=['accuracy'],
)

## 8. Early stopping

Watch the validation loss. If it does not improve for 5 epochs, stop, and go back to the best weights.

In [12]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
early_stopping

## 9. Train

20% of the training data is kept aside for validation: 20,000 reviews to learn from, 5,000 to check. This takes a few minutes per epoch on a CPU; use Google Colab with a GPU if your laptop is slow.

In [13]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 52ms/step - accuracy: 0.6146 - loss: 0.6396 - val_accuracy: 0.7234 - val_loss: 0.5397
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.7857 - loss: 0.4582 - val_accuracy: 0.7666 - val_loss: 0.5138
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 56ms/step - accuracy: 0.8436 - loss: 0.3636 - val_accuracy: 0.7408 - val_loss: 0.5609
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 31s 49ms/step - accuracy: 0.8831 - loss: 0.2891 - val_accuracy: 0.7060 - val_loss: 0.6927
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 54ms/step - accuracy: 0.9057 - loss: 0.2414 - val_accuracy: 0.7354 - val_loss: 0.6931
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 45s 72ms/step - accuracy: 0.9366 - loss: 0.1693 - val_accuracy: 0.7362 - val_loss: 0.7807
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 50s 80ms/step - accuracy: 0.9582 - loss: 0.1197 - val_accuracy: 0.7236 - val_loss: 0.8982


## 10. Check on the test set

25,000 reviews the model has never seen.

In [14]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc:.3f}')

782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.7697 - loss: 0.5046
Test accuracy: 0.770


## 11. Save the model

In [15]:
model.save('simple_rnn_imdb.keras')

## Experiment: relu instead of tanh

Build the same model with `SimpleRNN(128, activation='relu')` and **without** `clipnorm`. Watch the training loss: it can suddenly jump to huge values (exploding gradient). Then add `clipnorm=1.0` back and compare.